# Hate Speech Detection — Full Pipeline Demo

Demonstrates the complete 3-layer pipeline:

**Layer 1 (Retriever)** — SBERT encodes the query and retrieves semantically similar precedents from a FAISS index  
**Layer 2 (RAC Classifier)** — A fine-tuned RoBERTa classifier predicts *hate* / *not hate* from the augmented input  
**Layer 3 (Explainer)** — LLaMA 3.3 generates a grounded, human-readable explanation anchored to the retrieved evidence

## 1. Imports

In [9]:
import sys, psutil
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))  # adds src/ so retriever.py and llm_explainer.py are importable
from llm_explainer import (
    Layer2Output, ExplainerOutput,
    load_pipeline, run_all, display_result, wrap,
)

## Layer 3 — LLM Backend

Choose your LLM backend. Groq is recommended (free tier, fast). Uncomment Option B or C for Ollama or OpenAI.

In [ ]:
# --- Option A: Groq (free, recommended) ---
import os
from groq import Groq

llm_client = Groq(api_key='your_api_key_here')  # replace with your actual API key
LLM_MODEL  = "llama-3.3-70b-versatile"


print(f"LLM backend ready: {LLM_MODEL}")

LLM backend ready: llama-3.3-70b-versatile


## Pipeline Configuration

Choose the classifier model, index split and training dataset. `INDEX_SPLIT` and `DATASET` must match the saved weights.

In [11]:
MODEL_FAMILY = "roberta"   # "bert" | "roberta"
INDEX_SPLIT  = "full"      # "example" | "knowledge" | "full"
DATASET      = "ISHate"    # "IHC" | "ISHate"
K            = 3           # max neighbors to retrieve
THRESHOLD    = 0.3         # min cosine similarity

ROOT_DIR        = Path("../..")
INDEX_DIR       = ROOT_DIR / "corpus" / "index"
WEIGHTS_RAC_DIR = ROOT_DIR / "weights" / "weights_rac_best_hyperparameters"

## Input Texts

Add the texts you want to analyse. No ground-truth labels needed.

In [12]:
TEXTS = [
    {"id": 1, "text": "You will be drafted to defend zog"},
    # Add more entries as needed.
]

print(f"Loaded {len(TEXTS)} custom input(s).")
for t in TEXTS:
    print(f"  [{t['id']}] {t['text'][:100]}")

Loaded 1 custom input(s).
  [1] You will be drafted to defend zog


## Load Pipeline Components

Load the **SBERT retriever** (Layer 1), the **FAISS index**, and the **RAC classifier** (Layer 2).

In [13]:
ret_model, ret_tokenizer, index, documents, clf_model, clf_tokenizer, device = load_pipeline(
    MODEL_FAMILY, INDEX_SPLIT, DATASET, INDEX_DIR, WEIGHTS_RAC_DIR
)

Device: cpu
Loading retriever: sentence-transformers/all-mpnet-base-v2 ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15619.05it/s]


Loading index: ../../corpus/index/vdb_full.faiss ...
  Index size: 108,816 vectors
Loading RAC classifier: ../../weights/weights_rac_best_hyperparameters/roberta/sbert/full/ISHate ...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 17944.60it/s]


Ready: ROBERTA | index=sbert/full | trained_on=ISHate


## Layer 2 → Layer 3: Run the Full Pipeline

For each input text:
1. **Layer 1** — encode with SBERT and retrieve the top-*k* nearest neighbors from the FAISS index  
2. **Layer 2** — feed the augmented input (query + retrieved passages) to the RAC classifier  
3. **Layer 3** — pass the classification result and retrieved evidence to LLaMA for a grounded explanation

In [14]:
print(f"RAM available : {psutil.virtual_memory().available / 1e9:.1f} GB")
print(f"Config        : {MODEL_FAMILY.upper()} | index=sbert/{INDEX_SPLIT} | trained_on={DATASET}")

records = run_all(
    TEXTS, ret_model, ret_tokenizer, index, documents,
    clf_model, clf_tokenizer, device,
    llm_client, LLM_MODEL, K, THRESHOLD,
)

RAM available : 4.2 GB
Config        : ROBERTA | index=sbert/full | trained_on=ISHate
Extracting index vectors for numpy search... done  shape=(108816, 768)
[01/1] id=1  ... 

Encoding: 100%|██████████| 1/1 [00:00<00:00, 14.01it/s]


pred=HATE      conf=99.2%  0.8s

Done. 1 input(s) processed  |  total=0.0 min


## Results — Detailed Walkthrough

Full output for each input: the original text, Layer 2 classification with retrieved neighbors, and the Layer 3 LLM explanation.

In [15]:
for r in records:
    print("=" * 80)
    print(wrap(f"[{r['id']}]  TEXT : {r['text']}"))
    print("-" * 80)
    print(f"LAYER 2  : {r['predicted'].upper()}  ({r['confidence']:.1%} confidence)")
    print()
    print(f"RETRIEVED NEIGHBORS ({r['n_retrieved']}):")
    for i, p in enumerate(r["retrieved_passages"], 1):
        line = f"  [{i}] {p['score']:.4f}  [{p['label']}]  {p['text']}"
        print(wrap(line, indent=14))
    print()
    print("LAYER 3 EXPLANATION:")
    print(wrap(f"  Summary   : {r['summary']}"))
    print(f"  Severity  : {r['severity']}")
    print(f"  Action    : {r['action']}")
    targets = ", ".join(r["target_groups"]) if r["target_groups"] else "—"
    print(wrap(f"  Targets   : {targets}"))
    evidence = ", ".join(str(e) for e in r["evidence_used"]) if r["evidence_used"] else "—"
    print(wrap(f"  Evidence  : {evidence}"))
    if r["moderator_note"]:
        print(wrap(f"  Note      : {r['moderator_note']}"))
    valid_str = "✓ passed" if r["validation_passed"] else "✗ FAILED (forced human-review)"
    print(f"  Validation: {valid_str}")
    print("=" * 80)
    print()

[1]  TEXT : You will be drafted to defend zog
--------------------------------------------------------------------------------
LAYER 2  : HATE  (99.2% confidence)

RETRIEVED NEIGHBORS (3):
  [1] 0.4454  [hate]  zog: Zionist Occupation Government, an organization
              hypothesized by white supremacists to rule the world. Target
              category: Judaism.
  [2] 0.4274  [hate]  this is antifa . zog will take anyone to fight for greater
              israel
  [3] 0.4191  [hate]  Maximum resistance towards Zog !

LAYER 3 EXPLANATION:
  Summary   : The content was flagged as hate speech due to the reference to
              'zog', a term associated with white supremacist ideology and anti-
              Semitic conspiracy theories, as seen in evidence passages [1],
              [2], and [3].
  Severity  : high
  Action    : auto-block
  Targets   : Jewish people
  Evidence  : 1, 2, 3
  Note      : Review for potential white supremacist or anti-Semitic content
  Validation: ✓ 